<a href="https://colab.research.google.com/github/anshikaguptacs24-del/BIO_INSPIRED_LAB/blob/main/LAB_1_GENETIC_ALGORITHM_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import math

# ============================================================
# GENETIC ALGORITHM FOR TSP WITH REPLENISHMENT ARCS
# Based on the RATSP model from the research paper
# ============================================================

# ------------------------------------------------------------
# 1. PARAMETERS
# ------------------------------------------------------------

NUM_CITIES = 40
SPEED = 60.0              # km/hour
TMAX = 10 * 60            # 10 hours = 600 minutes
REST_TIME = 10 * 60       # 10 hours = 600 minutes

POPULATION_SIZE = 100
GENERATIONS = 300

CROSSOVER_RATE = 0.90
MUTATION_RATE = 0.10

random.seed(42)


# ------------------------------------------------------------
# 2. CITY DATA
# ------------------------------------------------------------
# Replace these with the 40 city coordinates from Table 1
# of the research paper.

cities = {
    1: (10, 20),
    2: (15, 45),
    3: (30, 25),
    4: (40, 60),
    5: (55, 20),
    6: (70, 45),
    7: (85, 30),
    8: (25, 75),
    9: (50, 80),
    10: (75, 70),

    11: (90, 15),
    12: (20, 10),
    13: (35, 40),
    14: (60, 65),
    15: (80, 55),
    16: (95, 45),
    17: (10, 90),
    18: (45, 95),
    19: (65, 90),
    20: (90, 85),

    21: (5, 55),
    22: (18, 65),
    23: (32, 15),
    24: (48, 35),
    25: (62, 15),
    26: (78, 10),
    27: (88, 60),
    28: (12, 35),
    29: (42, 70),
    30: (68, 35),

    31: (82, 90),
    32: (28, 90),
    33: (58, 95),
    34: (72, 80),
    35: (98, 70),
    36: (5, 75),
    37: (38, 5),
    38: (55, 50),
    39: (75, 25),
    40: (95, 95)
}


# Intracity travel times.
# Replace with the exact values from Table 1 if available.
intracity_time = {
    city: random.uniform(20, 100)
    for city in cities
}


# ------------------------------------------------------------
# 3. DISTANCE
# ------------------------------------------------------------

def distance(city1, city2):

    x1, y1 = cities[city1]
    x2, y2 = cities[city2]

    return math.sqrt(
        (x1 - x2) ** 2 +
        (y1 - y2) ** 2
    )


# ------------------------------------------------------------
# 4. TRAVEL TIME
# ------------------------------------------------------------

def travel_time(city1, city2):

    d = distance(city1, city2)

    # SPEED is km/hour.
    # Convert hours to minutes.
    return (d / SPEED) * 60


# ------------------------------------------------------------
# 5. CALCULATE ROUTE
# ------------------------------------------------------------

def evaluate_route(route):

    total_travel_time = 0
    cumulative_time = 0
    rest_count = 0

    # Start from city 1
    current_city = 1

    for next_city in route:

        # Travel from current city to next city
        t = travel_time(current_city, next_city)

        # Time spent inside next city
        city_time = intracity_time[next_city]

        # Check whether travelling + city visit
        # would exceed Tmax.

        if cumulative_time + t > TMAX:

            # Rest before travelling to next city
            rest_count += 1
            cumulative_time = 0

        cumulative_time += t

        # Check intracity travel
        if cumulative_time + city_time > TMAX:

            # Rest after entering the city
            rest_count += 1
            cumulative_time = 0

            cumulative_time += city_time

        else:
            cumulative_time += city_time

        total_travel_time += t
        current_city = next_city

    # Return to starting city
    t = travel_time(current_city, 1)

    if cumulative_time + t > TMAX:

        rest_count += 1
        cumulative_time = 0

    total_travel_time += t

    # Add intracity travel time
    total_intracity = sum(intracity_time.values())

    # Total objective
    total_time = (
        total_travel_time
        + total_intracity
        + rest_count * REST_TIME
    )

    return total_time, rest_count


# ------------------------------------------------------------
# 6. FITNESS
# ------------------------------------------------------------

def fitness(route):

    total_time, _ = evaluate_route(route)

    return 1 / total_time


# ------------------------------------------------------------
# 7. INITIAL POPULATION
# ------------------------------------------------------------

def create_population():

    population = []

    cities_to_visit = list(range(2, NUM_CITIES + 1))

    for _ in range(POPULATION_SIZE):

        chromosome = cities_to_visit.copy()

        random.shuffle(chromosome)

        population.append(chromosome)

    return population


# ------------------------------------------------------------
# 8. TOURNAMENT SELECTION
# ------------------------------------------------------------

def selection(population):

    tournament = random.sample(population, 3)

    winner = max(
        tournament,
        key=fitness
    )

    return winner.copy()


# ------------------------------------------------------------
# 9. ORDERED CROSSOVER (OX)
# ------------------------------------------------------------

def crossover(parent1, parent2):

    if random.random() > CROSSOVER_RATE:

        return parent1.copy(), parent2.copy()

    size = len(parent1)

    start, end = sorted(
        random.sample(range(size), 2)
    )

    child1 = [None] * size
    child2 = [None] * size

    # Copy selected section
    child1[start:end] = parent1[start:end]
    child2[start:end] = parent2[start:end]

    # Fill remaining positions
    def fill_child(child, parent):

        current = end

        for city in parent:

            if city not in child:

                if current >= size:
                    current = 0

                child[current] = city
                current += 1

    fill_child(child1, parent2)
    fill_child(child2, parent1)

    return child1, child2


# ------------------------------------------------------------
# 10. MUTATION
# ------------------------------------------------------------

def mutation(route):

    if random.random() < MUTATION_RATE:

        i, j = random.sample(
            range(len(route)), 2
        )

        route[i], route[j] = (
            route[j],
            route[i]
        )

    return route


# ------------------------------------------------------------
# 11. GENETIC ALGORITHM
# ------------------------------------------------------------

def genetic_algorithm():

    population = create_population()

    best_route = None
    best_time = float("inf")
    best_rests = 0

    for generation in range(GENERATIONS):

        # Sort according to fitness
        population.sort(
            key=fitness,
            reverse=True
        )

        # Best chromosome of generation
        current_best = population[0]

        current_time, current_rests = (
            evaluate_route(current_best)
        )

        # Update global best
        if current_time < best_time:

            best_time = current_time
            best_route = current_best.copy()
            best_rests = current_rests

        # Elitism
        new_population = [
            population[0].copy()
        ]

        # Generate remaining population
        while len(new_population) < POPULATION_SIZE:

            parent1 = selection(population)
            parent2 = selection(population)

            child1, child2 = crossover(
                parent1,
                parent2
            )

            child1 = mutation(child1)
            child2 = mutation(child2)

            new_population.append(child1)

            if len(new_population) < POPULATION_SIZE:
                new_population.append(child2)

        population = new_population

        # Display progress
        if (generation + 1) % 50 == 0:

            print(
                f"Generation {generation + 1}: "
                f"Best Time = {best_time:.2f} min, "
                f"Rests = {best_rests}"
            )

    return best_route, best_time, best_rests


# ------------------------------------------------------------
# 12. RUN GA
# ------------------------------------------------------------

best_route, best_time, best_rests = genetic_algorithm()


# ------------------------------------------------------------
# 13. FINAL OUTPUT
# ------------------------------------------------------------

print("\n==============================")
print("GENETIC ALGORITHM RESULT")
print("==============================")

print("Best Route:")
print("1 ->", " -> ".join(map(str, best_route)), "-> 1")

print(f"\nTotal Travel Time: {best_time:.2f} minutes")

print(f"Number of Rest Periods: {best_rests}")

print(
    f"Total Rest Time: "
    f"{best_rests * REST_TIME:.2f} minutes"
)

print(
    f"Fitness: "
    f"{1 / best_time:.8f}"
)

Generation 50: Best Time = 6440.91 min, Rests = 5
Generation 100: Best Time = 6320.29 min, Rests = 5
Generation 150: Best Time = 6249.12 min, Rests = 5
Generation 200: Best Time = 6223.25 min, Rests = 5
Generation 250: Best Time = 6223.25 min, Rests = 5
Generation 300: Best Time = 6208.20 min, Rests = 5

GENETIC ALGORITHM RESULT
Best Route:
1 -> 12 -> 23 -> 37 -> 25 -> 26 -> 5 -> 24 -> 28 -> 4 -> 29 -> 32 -> 18 -> 9 -> 15 -> 35 -> 20 -> 40 -> 31 -> 10 -> 34 -> 33 -> 19 -> 14 -> 27 -> 39 -> 30 -> 6 -> 16 -> 7 -> 11 -> 38 -> 8 -> 22 -> 36 -> 17 -> 21 -> 2 -> 13 -> 3 -> 1

Total Travel Time: 6208.20 minutes
Number of Rest Periods: 5
Total Rest Time: 3000.00 minutes
Fitness: 0.00016108
